# 第12周：动画、音效、状态管理、难度递增（Day 78-84）

这是一个可以自动检测练习题的 Jupyter Notebook，配合语音讲解使用。

**这个notebook每一天都包含三部分：🔊 语音讲解 + 📖 知识点讲解文字版 + ✏️ 练习题与🔍自动检测。**

**怎么用这个notebook：**
1. 从上到下按顺序运行每一个cell（选中cell后按 `Shift + Enter`）
2. 先点开语音讲解听一听，再看文字版讲解加深理解
3. 看到 `✏️ 你的代码` 的cell，把 `pass` 删掉，写上你的代码
4. 写完之后，运行下面紧跟着的 `🔍 检测` cell，看看是不是显示 ✅（通过）
5. 如果显示 ❌，看看提示信息，回去改一改代码再重新运行这两个cell（先改代码的cell，再改检测的cell）

**一个小提醒：** 这周涉及到动画、音效这些"看得见、听得到"的效果，在这个notebook里没办法真的放音乐或者看动画播放（而且这个环境没有真实的音响设备），所以还是用之前几周的办法——把每个效果拆成可以用具体数值验证的逻辑（比如"音量有没有被限制在0-1之间""第几帧该切换到哪张图""文字有没有被画到画布上"），核心知识点和真实写法完全一样。这周的Checkpoint会把第10-12周学的所有东西拼成一个完整的游戏原型，做完之后非常建议在自己电脑上真的跑一跑、听一听、玩一玩！

这个notebook不会给你答案，只会告诉你做得对不对，加油！

## 📑 本周内容导航

每天的内容都是先有 **🔊 语音讲解 + 📖 知识点讲解**，再有 **✏️ 练习 + 🔍 检测**，请按顺序学习：

- Day 78：动画基础——让静止的图案动起来
- Day 79：音效基础
- Day 80：状态管理——游戏的几种模式
- Day 81：难度递增
- Day 82：UI反馈——血条与分数显示
- Day 83：综合——把状态管理和难度递增结合起来
- Day 84：第12周 Checkpoint

In [ ]:
# 🔧 工具cell：检测练习题用的工具函数和pygame初始化，不需要修改，直接运行一次就好
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # 让pygame可以在没有真实显示器的环境里运行（这个notebook专用，自己电脑上写游戏不需要这一行）
os.environ["SDL_AUDIODRIVER"] = "dummy"  # 同理，没有真实音响设备时用这个代替
import pygame
import struct
import random
pygame.init()
pygame.mixer.init()
pygame.font.init()

def test_check(fn, desc):
    """fn 是一个不带参数的函数（通常是 lambda），运行它并根据结果打印 ✅ 或 ❌"""
    try:
        result = fn()
    except Exception as e:
        print(f"❌ {desc} —— 运行出错：{type(e).__name__}: {e}")
        return False
    ok = bool(result)
    if ok:
        print(f"✅ {desc}")
    else:
        print(f"❌ {desc}")
    return ok

print("工具函数准备好了，可以开始做练习题了！")

## Day 78：动画基础——让静止的图案动起来

**学习目标**：理解帧动画的基本原理，掌握怎么控制画面每隔几帧切换一次图案

### 📖 知识点讲解

**讲解要点**

- **游戏里的"动画"，本质上是快速切换一张一张不同的静止图片**，只要换得够快，人眼就会觉得是一个连续的动作。
  **实际场景**：小时候玩过的"翻页动画书"，每一页画的姿势都有一点点不一样，快速翻页的时候就会看起来像在动——游戏里的角色走路、跑步动画，原理完全一样。

- **如果每一帧都切换图案，动画会切换得太快、看不清楚**，所以通常会设置"每隔几帧才切换一次"，用一个计数器记录"已经过了多少帧"，数到设定的数字就切换到下一张图，然后计数器归零重新数。
  **实际场景**：红绿灯不是每一秒都在变色，而是绿灯亮一段固定的时间之后才变黄、变红——"攒够一定时间才切换"和"攒够一定帧数才切换动画"是同一种节奏控制思路。

- **切换到"下一张图"的时候，如果已经是最后一张了，需要回到第一张重新开始循环**，这种"数到头之后自动归零重新开始"的计算，可以用取余数（`%`）运算很简洁地实现：`(当前序号 + 1) % 总张数`。
  **实际场景**：时钟的秒针走到60秒，会回到0秒重新开始，而不是变成61秒——取余数运算刚好能描述这种"循环"。

In [1]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day78.mp3")

### ✏️ 练习1：`get_next_frame_index(current_index, total_frames)`

接收当前帧的序号`current_index`和总共有多少帧`total_frames`，用取余数运算算出"下一帧的序号"（到达最后一帧后要回到第0帧），`return`这个序号。

In [ ]:
# ✏️ 你的代码
def get_next_frame_index(current_index, total_frames):
    pass

In [ ]:
# 🔍 检测
results_78_1 = []
results_78_1.append(test_check(lambda: get_next_frame_index(0, 3) == 1, "正常情况下帧序号加1"))
results_78_1.append(test_check(lambda: get_next_frame_index(2, 3) == 0, "到达最后一帧后会循环回到第0帧"))
print(f"\n通过 {sum(results_78_1)}/{len(results_78_1)} 项")

### ✏️ 挑战：`make_animated_sprite(x, y, images, frames_per_switch)`

在函数内部定义一个继承`pygame.sprite.Sprite`的类（比如叫`AnimatedSprite`），保存`images`（一个图像列表）、`frames_per_switch`（每隔几帧切换一次）、`frame_index=0`（当前第几张）、`counter=0`（计数器），`self.image`设成`images[0]`，设置好`rect`并定位到`(x, y)`。再定义`update(self)`方法：`counter`加1，如果`counter`达到了`frames_per_switch`，就把`counter`归零，用Day78练习1的思路算出下一张图的序号，更新`frame_index`和`self.image`。创建一个这个类的对象并`return`。

In [ ]:
# ✏️ 你的代码
def make_animated_sprite(x, y, images, frames_per_switch):
    pass

In [ ]:
# 🔍 检测
results_78_2 = []
try:
    _img_a = pygame.Surface((10, 10)); _img_a.fill((255, 0, 0))
    _img_b = pygame.Surface((10, 10)); _img_b.fill((0, 255, 0))
    _anim = make_animated_sprite(5, 5, [_img_a, _img_b], 3)
    results_78_2.append(test_check(lambda: _anim.image.get_at((1, 1)) == (255, 0, 0), "刚创建时显示第一张图"))
    _anim.update(); _anim.update()
    results_78_2.append(test_check(lambda: _anim.image.get_at((1, 1)) == (255, 0, 0), "没到切换的帧数之前不会切换"))
    _anim.update()
    results_78_2.append(test_check(lambda: _anim.image.get_at((1, 1)) == (0, 255, 0), "到达切换的帧数后换成第二张图"))
    results_78_2.append(test_check(lambda: _anim.rect.topleft == (5, 5), "切换动画时位置不受影响"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_78_2.extend([False, False, False, False])
print(f"\n通过 {sum(results_78_2)}/{len(results_78_2)} 项")

## Day 79：音效基础

**学习目标**：理解pygame怎么播放声音，掌握用pygame.mixer.Sound和音量控制

### 📖 知识点讲解

**讲解要点**

- **`pygame.mixer`是pygame里专门管"声音"的模块**，要先调用`pygame.mixer.init()`初始化它，然后用`pygame.mixer.Sound(文件路径)`加载一个声音文件，得到一个"声音对象"，调用它的`.play()`方法就能播放。
  **实际场景**：很多游戏里，角色跳跃、受伤、捡到金币都会配一个短促的音效——这些音效文件通常在游戏一开始启动的时候就被提前加载好，播放的时候才不会有延迟卡顿。

- **每个声音对象都有自己的音量，可以用`.set_volume(数值)`设置，数值范围是0.0（无声）到1.0（最大音量）**，用`.get_volume()`可以查询当前的音量。如果设置的数值超出0到1的范围，应该先把它"卡"在这个范围内——又是边界检查技巧（`max(0.0, min(1.0, 数值))`）。
  **实际场景**：游戏里的"音效音量"滑条，拖到最左边是完全无声，拖到最右边是最大音量，不管玩家怎么拖，音量数值永远会被限制在这个合理范围内。

In [2]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day79.mp3")

### ✏️ 练习1：`make_sound_from_buffer(buffer)`

接收一段音频数据`buffer`（这个notebook里用`struct`生成，自己电脑上写游戏通常是用`pygame.mixer.Sound(文件路径)`读取一个真实的音频文件），用`pygame.mixer.Sound(buffer=buffer)`创建一个声音对象并`return`。

In [ ]:
# ✏️ 你的代码
def make_sound_from_buffer(buffer):
    pass

In [ ]:
# 🔍 检测
results_79_1 = []
_buf = struct.pack("<" + "h" * 100, *([0] * 100))
_snd1 = make_sound_from_buffer(_buf)
results_79_1.append(test_check(lambda: isinstance(_snd1, pygame.mixer.Sound) and _snd1.get_volume() == 1.0, "创建的Sound默认音量是1.0"))
print(f"\n通过 {sum(results_79_1)}/{len(results_79_1)} 项")

### ✏️ 挑战：`set_clamped_volume(sound, volume)`

接收一个声音对象`sound`和想设置的音量`volume`，先把`volume`限制在0.0到1.0之间（边界检查），再用`sound.set_volume(...)`设置这个限制后的音量，最后`return sound.get_volume()`。

In [ ]:
# ✏️ 你的代码
def set_clamped_volume(sound, volume):
    pass

In [ ]:
# 🔍 检测
results_79_2 = []
_snd2 = make_sound_from_buffer(_buf)
results_79_2.append(test_check(lambda: set_clamped_volume(_snd2, 0.5) == 0.5, "正常范围内的音量设置正确"))
results_79_2.append(test_check(lambda: set_clamped_volume(_snd2, 1.5) == 1.0, "超过1的音量会被限制在1.0"))
results_79_2.append(test_check(lambda: set_clamped_volume(_snd2, -0.2) == 0.0, "小于0的音量会被限制在0.0"))
print(f"\n通过 {sum(results_79_2)}/{len(results_79_2)} 项")

## Day 80：状态管理——游戏的几种模式

**学习目标**：理解游戏通常会有哪几种状态(开始界面/游戏中/结束界面)，掌握用状态字符串和事件来控制状态切换

### 📖 知识点讲解

**讲解要点**

- **一个完整的游戏通常至少有三种"状态"：`menu`（开始界面）、`playing`（正在玩）、`game_over`（结束界面）**，游戏在任意时刻只处于其中一种状态，不同状态下该做的事情完全不同。
  **实际场景**：手机上的音乐播放器也有类似的"状态"——"暂停中""播放中""加载中"，播放器在不同状态下，点击同一个按钮做的事情可能完全不一样。

- **状态的切换，通常是"当前状态+发生的某个事件 → 决定下一个状态是什么"**，比如"当前是`menu`状态，发生了'玩家点击了开始'这个事件，就切换到`playing`状态"。如果发生的事件跟当前状态"不匹配"，通常应该保持状态不变，忽略这个不合理的事件。
  **实际场景**：电梯门"关闭中"的状态下，如果有人按了"开门"按钮，电梯会重新打开（事件匹配，状态切换）；但如果电梯已经在"上升中"，这时候按"开门"是没有反应的（事件和当前状态不匹配，状态不会乱切换）。

In [3]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day80.mp3")

### ✏️ 练习1：`get_next_state(current_state, event)`

接收当前状态`current_state`和发生的事件`event`。如果当前是`"menu"`且事件是`"start"`，`return "playing"`；如果当前是`"playing"`且事件是`"game_over"`，`return "game_over"`；如果当前是`"game_over"`且事件是`"restart"`，`return "menu"`；其他所有情况，`return current_state`（状态不变）。

In [ ]:
# ✏️ 你的代码
def get_next_state(current_state, event):
    pass

In [ ]:
# 🔍 检测
results_80_1 = []
results_80_1.append(test_check(lambda: get_next_state("menu", "start") == "playing", "menu状态收到start事件后变成playing"))
results_80_1.append(test_check(lambda: get_next_state("playing", "game_over") == "game_over", "playing状态收到game_over事件后变成game_over"))
results_80_1.append(test_check(lambda: get_next_state("menu", "game_over") == "menu", "不合理的事件不会改变状态"))
print(f"\n通过 {sum(results_80_1)}/{len(results_80_1)} 项")

### ✏️ 挑战：`run_state_machine(initial_state, events)`

接收初始状态`initial_state`和一个事件列表`events`，用循环依次把每个事件交给Day80练习1的思路处理，更新当前状态，`return`处理完所有事件之后的最终状态。

In [ ]:
# ✏️ 你的代码
def run_state_machine(initial_state, events):
    pass

In [ ]:
# 🔍 检测
results_80_2 = []
results_80_2.append(test_check(lambda: run_state_machine("menu", ["start", "game_over", "restart", "start"]) == "playing", "按顺序处理多个事件后最终状态正确"))
results_80_2.append(test_check(lambda: run_state_machine("menu", ["game_over"]) == "menu", "序列中包含不合理事件时状态保持不变"))
print(f"\n通过 {sum(results_80_2)}/{len(results_80_2)} 项")

## Day 81：难度递增

**学习目标**：理解游戏越玩越难背后的实现思路，掌握用等级/分数计算出当前应该用的速度

### 📖 知识点讲解

**讲解要点**

- **很多游戏会随着玩的时间变长或者分数变高，让游戏变得更难**，常见的做法是维护一个"等级"（`level`），分数每达到一定数量，等级就提高一级。
  **实际场景**：很多手机解谜游戏前几关很简单，玩到后面关卡，同样类型的关卡会要求更快的反应速度或者更复杂的操作——"等级越高、难度越高"几乎是所有游戏共同的设计思路。

- **等级可以直接从分数计算出来**：比如"每20分提升一级"，那"当前等级 = 分数 ÷ 20（整除）+ 1"。有了等级之后，速度也可以用等级算出来：比如"速度 = 基础速度 + (等级-1) × 每级增加的速度"。
  **实际场景**：很多会员制度里，"会员等级"是直接根据"累计消费金额"算出来的，不需要人工去手动调整每个人的等级。

In [4]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day81.mp3")

### ✏️ 练习1：`speed_for_level(base_speed, level, increase_per_level)`

接收基础速度`base_speed`、当前等级`level`、每升一级增加的速度`increase_per_level`，`return base_speed + (level - 1) * increase_per_level`。

In [ ]:
# ✏️ 你的代码
def speed_for_level(base_speed, level, increase_per_level):
    pass

In [ ]:
# 🔍 检测
results_81_1 = []
results_81_1.append(test_check(lambda: speed_for_level(20, 1, 10) == 20, "第1级的速度就是基础速度"))
results_81_1.append(test_check(lambda: speed_for_level(20, 3, 10) == 40, "等级越高速度越快"))
print(f"\n通过 {sum(results_81_1)}/{len(results_81_1)} 项")

### ✏️ 挑战：`level_for_score(score, score_per_level)`

接收当前分数`score`和"每多少分升一级"`score_per_level`，`return score // score_per_level + 1`。

In [ ]:
# ✏️ 你的代码
def level_for_score(score, score_per_level):
    pass

In [ ]:
# 🔍 检测
results_81_2 = []
results_81_2.append(test_check(lambda: level_for_score(0, 20) == 1, "初始分数对应第1级"))
results_81_2.append(test_check(lambda: level_for_score(45, 20) == 3, "分数达到一定程度后等级正确提升"))
print(f"\n通过 {sum(results_81_2)}/{len(results_81_2)} 项")

## Day 82：UI反馈——血条与分数显示

**学习目标**：掌握用比例计算实现血条的宽度，了解怎么用pygame.font在画面上显示文字

### 📖 知识点讲解

**讲解要点**

- **血条的宽度，应该跟"当前生命值占总生命值的比例"成正比**：计算公式是`血条当前宽度 = 血条满宽度 × (当前生命值 / 最大生命值)`，同时要注意当前生命值不应该超过最大值、也不应该低于0（边界检查）。
  **实际场景**：手机电量图标里的那根"电量条"，宽度就是严格按照"当前电量 ÷ 满电量"算出来的。

- **`pygame.font`模块可以在画面上显示文字**。先用`pygame.font.SysFont(字体, 大小)`创建一个"字体对象"，再用`字体对象.render(文字内容, True, 颜色)`把文字渲染成一块图像，最后用`画布.blit(文字图像, (x, y))`把这块图像贴到画布上指定的位置。
  **实际场景**：几乎所有游戏画面上方或者角落都会显示当前分数、关卡——这些文字信息本质上都是先"画成一张小图"，再贴到屏幕固定位置上。

In [5]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day82.mp3")

### ✏️ 练习1：`health_bar_width(current_hp, max_hp, bar_max_width)`

接收当前生命值`current_hp`、最大生命值`max_hp`、血条满宽度`bar_max_width`。先把`current_hp`限制在`0`到`max_hp`之间，再按比例算出血条当前应该显示的宽度（结果用`int()`转成整数），`return`这个宽度。

In [ ]:
# ✏️ 你的代码
def health_bar_width(current_hp, max_hp, bar_max_width):
    pass

In [ ]:
# 🔍 检测
results_82_1 = []
results_82_1.append(test_check(lambda: health_bar_width(50, 100, 200) == 100, "血量一半时血条宽度正确"))
results_82_1.append(test_check(lambda: health_bar_width(150, 100, 200) == 200, "血量超过最大值时血条宽度不超过满条"))
print(f"\n通过 {sum(results_82_1)}/{len(results_82_1)} 项")

### ✏️ 挑战：`render_score_text(surface, font, score, x, y, color)`

接收画布`surface`、字体对象`font`、分数`score`、位置`x`、`y`、颜色`color`。用`font.render(f"分数: {score}", True, color)`渲染出文字图像，再用`surface.blit(文字图像, (x, y))`贴到画布上，`return surface`。

In [ ]:
# ✏️ 你的代码
def render_score_text(surface, font, score, x, y, color):
    pass

In [ ]:
# 🔍 检测
results_82_2 = []
try:
    _surf82 = pygame.Surface((200, 100)); _surf82.fill((0, 0, 0))
    _font82 = pygame.font.SysFont(None, 24)
    render_score_text(_surf82, _font82, 10, 10, 10, (255, 255, 255))
    results_82_2.append(test_check(lambda: any(_surf82.get_at((x, y))[:3] != (0, 0, 0) for x in range(10, 60) for y in range(10, 30)), "分数文字被正确画到了画布上"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_82_2.append(False)
print(f"\n通过 {sum(results_82_2)}/{len(results_82_2)} 项")

## Day 83：综合——把状态管理和难度递增结合起来

**学习目标**：练习把这周前几天学的状态管理和难度递增整合成一个统一管理游戏进度的工具

### 📖 知识点讲解

**讲解要点**

- **一个真实的游戏，往往需要同时维护"现在是什么状态"和"现在是第几级、速度多快"这两件事**，把它们放在同一个"游戏管理器"里统一维护，比分散成一堆互相不知道对方存在的变量要清晰得多。
  **实际场景**：公司的员工管理系统，会把"这个人是在职还是离职"和"这个人现在是什么职级"放在同一份员工档案里管理，而不是分成两套互相不关联的系统。

- **设计这样的"管理器"时，可以把"状态切换"和"难度更新"都包装成管理器自己的方法**，比如`start()`（状态变成`playing`）、`add_score(分数)`（加分，同时根据新的分数重新计算等级和速度）、`end_game()`（状态变成`game_over`）。
  **实际场景**：汽车的"中控系统"把"当前是驻车/行驶/倒车模式"和"当前档位、油门响应灵敏度"都整合在同一套逻辑里管理，驾驶员换一次档，系统会自动联动调整好几项相关的设置。

In [6]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day83.mp3")

### ✏️ 练习1：`update_difficulty(score, score_per_level, base_speed, increase_per_level)`

把Day81学的两个计算结合起来：先用`level_for_score`算出等级，再用`speed_for_level`算出对应的速度，按`(等级, 速度)`的顺序`return`一个元组。

In [ ]:
# ✏️ 你的代码
def update_difficulty(score, score_per_level, base_speed, increase_per_level):
    pass

In [ ]:
# 🔍 检测
results_83_1 = []
results_83_1.append(test_check(lambda: update_difficulty(45, 20, 20, 10) == (3, 40), "根据分数计算出的等级和速度都正确"))
print(f"\n通过 {sum(results_83_1)}/{len(results_83_1)} 项")

### ✏️ 挑战：`make_game_manager(base_speed, increase_per_level, score_per_level)`

在函数内部定义一个类`GameManager`，`__init__`里保存传入的参数，设置初始状态`self.state = "menu"`、`self.score = 0`、`self.level = 1`、`self.speed = base_speed`。再定义三个方法：`start(self)`把`state`设成`"playing"`；`add_score(self, points)`把`points`加到`self.score`上，再用Day83练习1的思路重新计算`self.level`和`self.speed`；`end_game(self)`把`state`设成`"game_over"`。创建一个这个类的对象并`return`。

In [ ]:
# ✏️ 你的代码
def make_game_manager(base_speed, increase_per_level, score_per_level):
    pass

In [ ]:
# 🔍 检测
results_83_2 = []
try:
    _gm = make_game_manager(20, 10, 20)
    results_83_2.append(test_check(lambda: _gm.state == "menu", "新游戏管理器的初始状态是menu"))
    _gm.start()
    results_83_2.append(test_check(lambda: _gm.state == "playing", "调用start()后状态变成playing"))
    _gm.add_score(45)
    results_83_2.append(test_check(lambda: _gm.score == 45 and _gm.level == 3 and _gm.speed == 40, "加分后等级和速度都正确更新"))
    _gm.end_game()
    results_83_2.append(test_check(lambda: _gm.state == "game_over", "调用end_game()后状态变成game_over"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_83_2.extend([False, False, False, False])
print(f"\n通过 {sum(results_83_2)}/{len(results_83_2)} 项")

## Day 84：第12周 Checkpoint

### 小测验（口头回答，答案由家长核对，这里不自动检测）

1. 动画"每隔几帧才切换一张图"这个节奏控制，为什么需要一个计数器？如果每一帧都切换会发生什么？
2. 声音的音量为什么也需要做"边界检查"？如果不做会怎么样？
3. 游戏的三种常见状态是什么？状态切换的基本规则是"当前状态 + 什么 → 下一个状态"？
4. 游戏难度递增的等级，是怎么从分数计算出来的？等级又是怎么影响速度的？
5. 血条的宽度是怎么跟生命值关联起来的？

### 挑战题：`make_final_game(screen_w, screen_h, player_w, player_h, item_size, base_speed, points_each, score_per_level, increase_per_level, player_speed=15, max_misses=3)` —— 完成期末游戏的"可玩原型"

**背景**：这是第五、六阶段学到现在最重要的一次综合练习——把第10-12周攒下的所有"零件"（移动+边界检测、Sprite+精灵组+碰撞检测+计分、状态管理+难度递增）拼装成一个从开始到结束、有完整流程、而且会越玩越难的小游戏原型。

**要求**：在函数内部定义一个表示"游戏"的类（比如叫`GamePrototype`），至少包含：

1. **状态管理**：`self.state`初始是`"menu"`，`start(self)`方法能让状态变成`"playing"`
2. **玩家**：`self.player_rect`，初始在窗口底部中间；`move_player(self, pressed_keys)`方法：根据方向键左右移动，并做边界检测（参考第10周）
3. **掉落物**：`self.items`是一个精灵组；`spawn_item(self, x=None)`方法：用当前的`self.speed`创建一个掉落物（参考第11周的`FallingItem`写法），加入`self.items`
4. **难度与计分**：`self.score`、`self.level`、`self.speed`（初始分别是`0`、`1`、`base_speed`）；`step(self)`方法里，接住一个掉落物时，`self.score`加上`points_each`，然后用Day83练习1的思路**重新计算**`self.level`和`self.speed`（这是难度递增真正生效的地方——后面新生成的掉落物会使用更新后的`self.speed`）
5. **miss与游戏结束**：`step(self)`里，掉落物掉出窗口底部还没被接住时，`self.misses`加1；`self.misses`达到`max_misses`时，把`self.state`设成`"game_over"`（初始`self.misses = 0`）

函数最后创建并`return`一个`GamePrototype`对象。

**完成标准**
- [ ] 游戏能从`menu`状态正确开始（`start()`后进入`playing`）
- [ ] 碰撞检测、计分、miss判断都能正确触发，不会搞混
- [ ] 至少有一个难度递增机制确实生效（接住足够多道具后，新掉落物的下落速度确实变快了）
- [ ] miss次数达到上限后，游戏状态能正确变成`game_over`
- [ ] 完成之后，建议自己口头跟家长讲一遍这个游戏的完整规则：怎么开始、怎么得分、怎么变难、怎么结束

In [7]:
# 🔊 语音讲解（可以直接点击播放）
from IPython.display import Audio
Audio("audio/day84.mp3")

In [ ]:
# ✏️ 你的代码
def make_final_game(screen_w, screen_h, player_w, player_h, item_size, base_speed, points_each,
                     score_per_level, increase_per_level, player_speed=15, max_misses=3):
    pass

In [ ]:
# 🔍 检测
results_84 = []

try:
    _game1 = make_final_game(300, 200, 40, 10, 20, 20, 10, 20, 10, player_speed=15, max_misses=3)
    results_84.append(test_check(lambda: _game1.state == "menu" and _game1.score == 0 and _game1.level == 1 and _game1.speed == 20, "新游戏的初始状态正确"))
except Exception as e:
    print(f"❌ 初始状态测试运行出错：{type(e).__name__}: {e}")
    results_84.append(False)

try:
    _game1.start()
    results_84.append(test_check(lambda: _game1.state == "playing", "调用start()后游戏进入playing状态"))
except Exception as e:
    print(f"❌ start()测试运行出错：{type(e).__name__}: {e}")
    results_84.append(False)

try:
    for _ in range(2):
        _game1.spawn_item(x=130)
        for _ in range(9):
            _game1.step()
    results_84.append(test_check(lambda: _game1.score == 20 and _game1.level == 2 and _game1.speed == 30, "接住足够多道具后，等级和下落速度都提升了"))
    _item_after_levelup = _game1.spawn_item(x=130)
    results_84.append(test_check(lambda: _item_after_levelup.speed == 30, "等级提升后，新生成的掉落物下落速度确实变快了"))
except Exception as e:
    print(f"❌ 难度递增测试运行出错：{type(e).__name__}: {e}")
    results_84.extend([False, False])

try:
    _game2 = make_final_game(300, 200, 40, 10, 20, 20, 10, 20, 10, player_speed=15, max_misses=3)
    _game2.start()
    _game2.spawn_item(x=0)
    _game2.spawn_item(x=280)
    _game2.spawn_item(x=50)
    for _ in range(11):
        _game2.step()
    results_84.append(test_check(lambda: _game2.misses == 3 and _game2.state == "game_over", "miss次数达到上限后，游戏状态正确变成game_over"))
except Exception as e:
    print(f"❌ 游戏结束测试运行出错：{type(e).__name__}: {e}")
    results_84.append(False)

try:
    _game3 = make_final_game(300, 200, 40, 10, 20, 20, 10, 20, 10, player_speed=15, max_misses=3)
    _game3.move_player({pygame.K_RIGHT})
    results_84.append(test_check(lambda: _game3.player_rect.x == 145, "玩家正常移动正确"))
    _game3.player_rect.x = 255
    _game3.move_player({pygame.K_RIGHT})
    results_84.append(test_check(lambda: _game3.player_rect.x == 260, "玩家移动不会超出窗口边界"))
except Exception as e:
    print(f"❌ 玩家移动测试运行出错：{type(e).__name__}: {e}")
    results_84.extend([False, False])

print()
passed = sum(results_84); total = len(results_84)
print(f"第12周Checkpoint：通过 {passed}/{total} 项")
if passed == total:
    print("🎉 第12周完成！记得让家长核对一下小测验的口头回答，也记得口头讲一遍游戏规则。恭喜走完游戏开发核心技能这一阶段，接下来要进入期末大项目：完整游戏啦！")

---

恭喜完成第12周的所有练习，也恭喜做出了期末游戏的"可玩原型"！这是整个100天计划里最综合的一次练习，把好几周学的东西都用上了。如果有检测没通过，可以回去看看对应的"讲解要点"和"实际场景"部分，想一想问题出在哪里，也强烈建议在自己电脑上用真实的pygame窗口把这个原型跑起来，真正玩一玩、听一听、看一看它是什么样子。接下来两周就是期末大项目啦！